# EXO-200 v1 Transformer Results

Loads completed artifacts only. This notebook does not load HDF5 data, train models, or rerun inference. Higher ROC-AUC is better.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

configured_root = os.environ.get('EXO200_TRANSFORMER_PROJECT_ROOT')
candidate_roots = []
if configured_root:
    candidate_roots.append(Path(configured_root).expanduser())
candidate_roots.extend([
    Path.cwd() / 'exo200_detector', Path.cwd(),
    Path.cwd().parent / 'exo200_detector', Path.cwd().parent,
    Path.cwd().parent.parent / 'exo200_detector',
])
PROJECT_ROOT = next(
    (candidate.resolve() for candidate in candidate_roots
     if (candidate / 'exo_transformer').is_dir()
     and (candidate / 'exobench').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate exo200_detector; set EXO200_TRANSFORMER_PROJECT_ROOT.'
    )
OUTPUT_ROOT = Path(os.environ.get(
    'EXO200_OUTPUT_ROOT',
    str(PROJECT_ROOT / 'results' / 'transformer_official_v1'),
)).expanduser()
print('Reading results from:', OUTPUT_ROOT)

In [ ]:
EXPECTED_RUNS = [
    f'classification__{tokenization}__{position_encoding}'
    for tokenization in ('raw_patches', 'segment_summary', 'pulse_entities')
    for position_encoding in ('coordinate_mlp', 'fourier_coordinates')
]

def load_completed_runs(output_root):
    rows = []
    for run_id in EXPECTED_RUNS:
        path = output_root / run_id / 'run_summary.json'
        if path.is_file():
            rows.append(json.loads(path.read_text(encoding='utf-8')))
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(
        ['test_auc', 'best_validation_auc'], ascending=False
    ).reset_index(drop=True)

results = load_completed_runs(OUTPUT_ROOT)
completed = set(results['run_id']) if not results.empty else set()
status = pd.DataFrame({
    'run_id': EXPECTED_RUNS,
    'status': ['complete' if run_id in completed else 'incomplete'
               for run_id in EXPECTED_RUNS],
})
print(f'Completed: {len(completed)} / {len(EXPECTED_RUNS)}')
display(status)

In [ ]:
if results.empty:
    print('No completed EXO-200 Transformer runs yet.')
else:
    columns = [
        'tokenization', 'position_encoding',
        'best_validation_auc', 'test_auc', 'test_accuracy',
        'best_epoch', 'epochs_completed', 'minutes_per_epoch',
        'parameter_count', 'test_events',
    ]
    display(results[columns].style.format({
        'best_validation_auc': '{:.6f}',
        'test_auc': '{:.6f}',
        'test_accuracy': '{:.6f}',
        'minutes_per_epoch': '{:.3f}',
        'parameter_count': '{:,.0f}',
        'test_events': '{:,.0f}',
    }))

In [ ]:
if not results.empty:
    plot_data = results.copy()
    plot_data['representation'] = (
        plot_data['tokenization'] + '\n' + plot_data['position_encoding']
    )
    figure, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].bar(plot_data['representation'], plot_data['test_auc'])
    axes[0].set_title('Held-out test ROC-AUC')
    axes[0].set_ylabel('ROC-AUC (higher is better)')
    axes[0].tick_params(axis='x', rotation=35)
    axes[0].set_ylim(max(0.0, plot_data['test_auc'].min() - 0.03), 1.0)

    axes[1].bar(plot_data['representation'], plot_data['minutes_per_epoch'])
    axes[1].set_title('Training runtime')
    axes[1].set_ylabel('Minutes per epoch')
    axes[1].tick_params(axis='x', rotation=35)
    figure.tight_layout()
    plt.show()

In [ ]:
history_rows = []
for run_id in EXPECTED_RUNS:
    history_path = OUTPUT_ROOT / run_id / 'history.json'
    config_path = OUTPUT_ROOT / run_id / 'run_config.json'
    if not history_path.is_file() or not config_path.is_file():
        continue
    history = json.loads(history_path.read_text(encoding='utf-8'))
    config = json.loads(config_path.read_text(encoding='utf-8'))
    representation = config['representation']
    label = (
        representation['tokenization']['tokenization'] + ' / ' +
        representation['position_encoding']
    )
    for row in history:
        history_rows.append({
            'run_id': run_id,
            'label': label,
            'epoch': row['epoch'],
            'train_loss': row['train_loss'],
            'validation_loss': row['validation_loss'],
            'validation_auc': row['validation_metrics']['auc'],
        })

history_table = pd.DataFrame(history_rows)
if history_table.empty:
    print('No training histories available.')
else:
    figure, axes = plt.subplots(1, 2, figsize=(16, 5))
    for label, group in history_table.groupby('label'):
        axes[0].plot(group['epoch'], group['validation_auc'], label=label)
        axes[1].plot(group['epoch'], group['validation_loss'], label=label)
    axes[0].set(title='Validation ROC-AUC', xlabel='Epoch', ylabel='ROC-AUC')
    axes[1].set(title='Validation loss', xlabel='Epoch', ylabel='BCE loss')
    axes[0].legend(fontsize=8)
    axes[1].legend(fontsize=8)
    figure.tight_layout()
    plt.show()